<a href="https://colab.research.google.com/github/kl01abhinav2-coder/git_hub_1/blob/main/Data_Acquisition_casestudy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import requests
import sqlite3

1: Load SpaceX Launch Data from API

In [ ]:
url = "https://api.spacexdata.com/v4/launches"# Load SpaceX launch data from API
data = requests.get(url)
data1 = data.json()

df = pd.DataFrame(data1, columns=["name", "date_utc", "success", "details", "rocket"])# Create a DataFrame with required columns
print(df)

                       name                  date_utc success  \
0                 FalconSat  2006-03-24T22:30:00.000Z   False   
1                   DemoSat  2007-03-21T01:10:00.000Z   False   
2               Trailblazer  2008-08-03T03:34:00.000Z   False   
3                    RatSat  2008-09-28T23:15:00.000Z    True   
4                  RazakSat  2009-07-13T03:35:00.000Z    True   
..                      ...                       ...     ...   
200           Transporter-6  2022-12-01T00:00:00.000Z    None   
201                   TTL-1  2022-12-01T00:00:00.000Z    None   
202  WorldView Legion 1 & 2  2022-12-01T00:00:00.000Z    None   
203     Viasat-3 & Arcturus  2022-12-01T00:00:00.000Z    None   
204          O3b mPower 3.4  2022-12-01T00:00:00.000Z    None   

                                               details  \
0     Engine failure at 33 seconds and loss of vehicle   
1    Successful first stage burn and transition to ...   
2    Residual stage 1 thrust led to collision

In [ ]:
df["date_utc"] = pd.to_datetime(df["date_utc"])# Convert date_utc to datetime format
df["year"] = df["date_utc"].dt.year# Extract year from the date
print(df)

                       name                  date_utc success  \
0                 FalconSat 2006-03-24 22:30:00+00:00   False   
1                   DemoSat 2007-03-21 01:10:00+00:00   False   
2               Trailblazer 2008-08-03 03:34:00+00:00   False   
3                    RatSat 2008-09-28 23:15:00+00:00    True   
4                  RazakSat 2009-07-13 03:35:00+00:00    True   
..                      ...                       ...     ...   
200           Transporter-6 2022-12-01 00:00:00+00:00    None   
201                   TTL-1 2022-12-01 00:00:00+00:00    None   
202  WorldView Legion 1 & 2 2022-12-01 00:00:00+00:00    None   
203     Viasat-3 & Arcturus 2022-12-01 00:00:00+00:00    None   
204          O3b mPower 3.4 2022-12-01 00:00:00+00:00    None   

                                               details  \
0     Engine failure at 33 seconds and loss of vehicle   
1    Successful first stage burn and transition to ...   
2    Residual stage 1 thrust led to collision

2: Load Rocket Metadata

In [ ]:
url = "https://api.spacexdata.com/v4/rockets"
rocket_data = requests.get(url).json()
rocket_df = pd.DataFrame(rocket_data, columns=["id", "name", "type", "active", "stages"])

print(rocket_df)

                         id          name    type  active  stages
0  5e9d0d95eda69955f709d1eb      Falcon 1  rocket   False       2
1  5e9d0d95eda69973a809d1ec      Falcon 9  rocket    True       2
2  5e9d0d95eda69974db09d1ed  Falcon Heavy  rocket    True       2
3  5e9d0d96eda699382d09d1ee      Starship  rocket   False       2


3:  Merge Launch and Rocket Data

In [ ]:

merge_df = pd.DataFrame(df.merge(rocket_df, left_on="rocket", right_on="id"))
merge_df

,name_x,date_utc,success,details,rocket,year,id,name_y,type,active,stages
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
...,...,...,...,...,...,...,...,...,...,...,...
200,Transporter-6,2022-12-01 00:00:00+00:00,None,None,5e9d0d95eda69973a809d1ec,2022,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,True,2
201,TTL-1,2022-12-01 00:00:00+00:00,None,None,5e9d0d95eda69973a809d1ec,2022,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,True,2
202,WorldView Legion 1 & 2,2022-12-01 00:00:00+00:00,None,None,5e9d0d95eda69973a809d1ec,2022,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,True,2
203,Viasat-3 & Arcturus,2022-12-01 00:00:00+00:00,None,None,5e9d0d95eda69974db09d1ed,2022,5e9d0d95eda69974db09d1ed,Falcon Heavy,rocket,True,2


4:  Add Simulated Country Information


In [ ]:
countries = ["usa","russia","india","china","france"]
merge_df["countries"] = np.random.choice(countries,size=len(merge_df))
merge_df

,name_x,date_utc,success,details,rocket,year,id,name_y,type,active,stages,countries
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,india
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,russia
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,russia
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,china
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,usa
...,...,...,...,...,...,...,...,...,...,...,...,...
200,Transporter-6,2022-12-01 00:00:00+00:00,None,None,5e9d0d95eda69973a809d1ec,2022,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,True,2,france
201,TTL-1,2022-12-01 00:00:00+00:00,None,None,5e9d0d95eda69973a809d1ec,2022,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,True,2,russia
202,WorldView Legion 1 & 2,2022-12-01 00:00:00+00:00,None,None,5e9d0d95eda69973a809d1ec,2022,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,True,2,france
203,Viasat-3 & Arcturus,2022-12-01 00:00:00+00:00,None,None,5e9d0d95eda69974db09d1ed,2022,5e9d0d95eda69974db09d1ed,Falcon Heavy,rocket,True,2,india


 5:  Store Merged Data in SQLite3


In [ ]:
conn = sqlite3.connect("spacex.db")
merge_df.to_sql("launches",conn,if_exists="replace",index=False)
conn.close()
merge_df

,name_x,date_utc,success,details,rocket,year,id,name_y,type,active,stages,countries
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,india
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,russia
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,russia
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,china
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,usa
...,...,...,...,...,...,...,...,...,...,...,...,...
200,Transporter-6,2022-12-01 00:00:00+00:00,None,None,5e9d0d95eda69973a809d1ec,2022,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,True,2,france
201,TTL-1,2022-12-01 00:00:00+00:00,None,None,5e9d0d95eda69973a809d1ec,2022,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,True,2,russia
202,WorldView Legion 1 & 2,2022-12-01 00:00:00+00:00,None,None,5e9d0d95eda69973a809d1ec,2022,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,True,2,france
203,Viasat-3 & Arcturus,2022-12-01 00:00:00+00:00,None,None,5e9d0d95eda69974db09d1ed,2022,5e9d0d95eda69974db09d1ed,Falcon Heavy,rocket,True,2,india


6:  Run SQL Queries on the Data to analyze

In [ ]:
conn = sqlite3.connect('spacex.db')

query = """
SELECT countries, COUNT(*) AS launch_count
FROM launches
GROUP BY countries
ORDER BY launch_count DESC;
"""
df_country = pd.read_sql_query(query, conn)
df_country


,countries,launch_count
0,russia,47
1,france,46
2,india,42
3,usa,36
4,china,34


In [ ]:
query = """
SELECT year, COUNT(*) AS launch_count
FROM launches
GROUP BY year
ORDER BY launch_count DESC
LIMIT 1;
"""
df_top_year = pd.read_sql_query(query, conn)
df_top_year

,year,launch_count
0,2022,62


In [ ]:

query = """
SELECT name_y, COUNT(*) AS launch_count
FROM launches
GROUP BY name_y
ORDER BY launch_count DESC
LIMIT 1;
"""
df_top_missions=pd.read_sql_query(query,conn)
df_top_missions

,name_y,launch_count
0,Falcon 9,195
